# Firestore 읽기량 분석 노트북

**목표**: Firestore에서 발생하는 read 비용의 원인을 파악한다.  
**원칙**: Firebase는 단 **1회만** 풀스캔(STEP 1)하고, 이후 모든 분석은 `dump/` 폴더의 로컬 JSON만 사용한다.

## 사용 순서
1. **사전 준비**
   - `pip install firebase-admin pandas matplotlib`
   - Firebase 콘솔 → 프로젝트 설정 → 서비스 계정 → 새 비공개 키 생성 → `serviceAccountKey.json`을 **프로젝트 루트**에 저장
   - `dump/`, `serviceAccountKey.json` 은 반드시 `.gitignore` 에 추가
2. **STEP 0~1 (1회만)**: 모든 컬렉션을 로컬 덤프
3. **STEP 2~** : 로컬 캐시 기반 분석 (Firebase read 0)

> 다시 분석할 때는 STEP 1을 건너뛰고 STEP 2부터 실행하면 됨. 갱신이 필요하면 `dump/` 폴더 삭제 후 다시 STEP 1 실행.

## STEP 0 — Setup

In [8]:
import json
import pathlib
import statistics
from collections import Counter, defaultdict
from datetime import datetime, timezone

import firebase_admin
from firebase_admin import credentials, firestore

# 노트북이 scripts/ 안에 있다고 가정 → 프로젝트 루트 = ..
ROOT = pathlib.Path('..').resolve()
SERVICE_ACCOUNT_PATH = pathlib.Path(r'C:\Users\User\Desktop\Project\work-manager\waff-work-manager\.claude\worktrees\romantic-goodall-e8427d\scripts\waff-work-manager-firebase-adminsdk-fbsvc-2cd6669f86.json')
DUMP_DIR = pathlib.Path('./dump').resolve()
DUMP_DIR.mkdir(exist_ok=True)

# 코드베이스에서 발견한 모든 top-level 컬렉션
# (lib/firestore-service*.ts, lib/gpt-test-*.ts 참고)
KNOWN_COLLECTIONS = [
    # 전략 (prefix 없음)
    'projects', 'tasks', 'history', 'settings',
    # FA
    'fa_projects', 'fa_tasks', 'fa_history', 'fa_settings',
    # ICT
    'ict_projects', 'ict_tasks', 'ict_history', 'ict_settings',
    # 사용자/권한
    'user_profiles', 'user_page_permissions',
    # GPT 관련
    'gpt_email_agent_settings', 'gpt_test_vector_stores',
]

print(f'ROOT             : {ROOT}')
print(f'SERVICE_ACCOUNT  : {SERVICE_ACCOUNT_PATH}  (exists={SERVICE_ACCOUNT_PATH.exists()})')
print(f'DUMP_DIR         : {DUMP_DIR}')
print(f'KNOWN_COLLECTIONS: {len(KNOWN_COLLECTIONS)}개')

ROOT             : C:\Users\User\Desktop\Project\work-manager\waff-work-manager\.claude\worktrees\romantic-goodall-e8427d
SERVICE_ACCOUNT  : C:\Users\User\Desktop\Project\work-manager\waff-work-manager\.claude\worktrees\romantic-goodall-e8427d\scripts\waff-work-manager-firebase-adminsdk-fbsvc-2cd6669f86.json  (exists=True)
DUMP_DIR         : C:\Users\User\Desktop\Project\work-manager\waff-work-manager\.claude\worktrees\romantic-goodall-e8427d\scripts\dump
KNOWN_COLLECTIONS: 16개


In [9]:
# firebase-admin 초기화 (이미 초기화돼 있으면 재사용)
if not firebase_admin._apps:
    cred = credentials.Certificate(str(SERVICE_ACCOUNT_PATH))
    firebase_admin.initialize_app(cred)
db = firestore.client()
print('Firestore client ready:', db.project)

Firestore client ready: waff-work-manager


## STEP 1 — 전체 컬렉션 로컬 덤프 (1회만 실행)

- `db.collections()` 로 root collection 목록을 실시간 조회 → 알려진 목록과 합쳐서 누락된 것 탐지
- 컬렉션별 `stream()` 으로 전체 문서 다운로드 → `dump/{collection}.json` 저장
- **이미 덤프된 파일은 스킵** (`force=True` 로 덮어쓰기)

> ⚠️ 이 셀이 실제로 Firebase read 를 소비하는 유일한 부분.

In [10]:
# 누락된 root collection 자동 발견
discovered = sorted([c.id for c in db.collections()])
missing = [c for c in discovered if c not in KNOWN_COLLECTIONS]
extra = [c for c in KNOWN_COLLECTIONS if c not in discovered]

print(f'Firestore root collections ({len(discovered)}):')
for c in discovered:
    print(f'  - {c}')
if missing:
    print(f'\n⚠️ KNOWN_COLLECTIONS 에 없는 컬렉션: {missing}')
if extra:
    print(f'\nℹ️ 코드에는 있지만 Firestore에는 없는 컬렉션: {extra}')

# 실제 덤프 대상 = Firestore에 존재하는 모든 컬렉션 (소실 방지)
TARGET_COLLECTIONS = sorted(set(KNOWN_COLLECTIONS) | set(discovered))
print(f'\n→ dump 대상: {len(TARGET_COLLECTIONS)}개')

Firestore root collections (19):
  - ai_email_work_proposals
  - dailyReportApplied
  - dailyReportRuns
  - fa_history
  - fa_projects
  - fa_settings
  - fa_tasks
  - gpt_email_agent_settings
  - gpt_test_vector_stores
  - history
  - ict_history
  - ict_projects
  - ict_settings
  - ict_tasks
  - projects
  - settings
  - tasks
  - user_page_permissions
  - user_profiles

⚠️ KNOWN_COLLECTIONS 에 없는 컬렉션: ['ai_email_work_proposals', 'dailyReportApplied', 'dailyReportRuns']

→ dump 대상: 19개


In [11]:
def serialize(v):
    """Firestore 값을 JSON-직렬화 가능한 형태로 변환."""
    if hasattr(v, 'isoformat'):  # datetime / DatetimeWithNanoseconds
        return {'__datetime__': v.isoformat()}
    if hasattr(v, 'path'):  # DocumentReference
        return {'__ref__': v.path}
    if hasattr(v, 'latitude') and hasattr(v, 'longitude'):  # GeoPoint
        return {'__geo__': [v.latitude, v.longitude]}
    if isinstance(v, dict):
        return {k: serialize(x) for k, x in v.items()}
    if isinstance(v, list):
        return [serialize(x) for x in v]
    if isinstance(v, bytes):
        return {'__bytes_len__': len(v)}
    return v

def dump_collection(name: str, force: bool = False) -> int:
    out = DUMP_DIR / f'{name}.json'
    if out.exists() and not force:
        existing = json.loads(out.read_text(encoding='utf-8'))
        print(f'  [skip ] {name:35s} {len(existing):>7,} docs  ({out.stat().st_size:>10,} B)')
        return len(existing)
    docs = []
    for d in db.collection(name).stream():
        docs.append({'id': d.id, 'data': serialize(d.to_dict() or {})})
    out.write_text(json.dumps(docs, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'  [dump ] {name:35s} {len(docs):>7,} docs  ({out.stat().st_size:>10,} B)')
    return len(docs)

# force=False → 이미 덤프된 컬렉션은 건너뜀
FORCE_REDUMP = False

total = 0
for c in TARGET_COLLECTIONS:
    total += dump_collection(c, force=FORCE_REDUMP)

meta = {
    'dumped_at': datetime.now(timezone.utc).isoformat(),
    'project': db.project,
    'collections': TARGET_COLLECTIONS,
    'total_documents': total,
}
(DUMP_DIR / '_meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
print(f'\n✅ total documents in dump: {total:,}')

  [dump ] ai_email_work_proposals                  10 docs  (    33,935 B)
  [dump ] dailyReportApplied                        1 docs  (       412 B)
  [dump ] dailyReportRuns                          36 docs  (   115,328 B)
  [dump ] fa_history                            1,880 docs  ( 3,071,854 B)
  [dump ] fa_projects                              18 docs  (     6,110 B)
  [dump ] fa_settings                               2 docs  (       417 B)
  [dump ] fa_tasks                                461 docs  (   284,877 B)
  [dump ] gpt_email_agent_settings                  1 docs  (       676 B)
  [dump ] gpt_test_vector_stores                    2 docs  (       754 B)
  [dump ] history                               7,075 docs  (10,876,214 B)
  [dump ] ict_history                              34 docs  (    33,988 B)
  [dump ] ict_projects                              6 docs  (     2,109 B)
  [dump ] ict_settings                              1 docs  (       298 B)
  [dump ] ict_tasks      

## STEP 2 — 로컬 캐시 로드

여기부터의 셀은 모두 로컬 JSON 만 사용 → 추가 Firebase read = **0**.

In [ ]:
def load(name: str) -> list[dict]:
    p = DUMP_DIR / f'{name}.json'
    if not p.exists():
        return []
    return json.loads(p.read_text(encoding='utf-8'))

meta = json.loads((DUMP_DIR / '_meta.json').read_text(encoding='utf-8'))
data = {c: load(c) for c in meta['collections']}

print(f"dump 시점: {meta['dumped_at']}")
print(f"프로젝트  : {meta['project']}")
print(f"총 문서  : {meta['total_documents']:,}")
print()
for c, docs in sorted(data.items(), key=lambda x: -len(x[1])):
    print(f'  {c:35s} {len(docs):>7,} docs')

## STEP 3 — 컬렉션별 문서 수 · 크기 분석

Firestore 과금은 **read 1건 = 문서 1개**.  
→ `getDocs(collection(...))` 식의 풀스캔이 한 번 발생하면 **그 컬렉션의 문서 수만큼 read가 청구**된다.  
→ 문서 수가 큰 컬렉션부터 **쿼리에 `where`/`limit`/페이징을 거는 것이 가장 효과 큼**.

In [ ]:
def doc_bytes(d: dict) -> int:
    return len(json.dumps(d['data'], ensure_ascii=False).encode('utf-8'))

rows = []
for c, docs in data.items():
    if not docs:
        rows.append((c, 0, 0, 0, 0, 0, 0))
        continue
    sizes = sorted(doc_bytes(d) for d in docs)
    rows.append((
        c,
        len(docs),
        sum(sizes) / 1024,                                # total KB
        statistics.mean(sizes),                           # avg B
        sizes[len(sizes) // 2],                           # median
        sizes[min(int(len(sizes) * 0.95), len(sizes)-1)], # p95
        sizes[-1],                                        # max
    ))

rows.sort(key=lambda r: -r[1])  # 문서 수 내림차순

header = f"{'collection':35s} {'docs':>8s} {'total KB':>12s} {'avg B':>8s} {'p50 B':>8s} {'p95 B':>8s} {'max B':>10s}"
print(header)
print('-' * len(header))
for c, n, kb, avg, p50, p95, mx in rows:
    print(f'{c:35s} {n:>8,} {kb:>12,.1f} {avg:>8,.0f} {p50:>8,.0f} {p95:>8,.0f} {mx:>10,.0f}')

In [ ]:
import matplotlib.pyplot as plt

nonempty = [r for r in rows if r[1] > 0]
names = [r[0] for r in nonempty]
counts = [r[1] for r in nonempty]
kbs = [r[2] for r in nonempty]

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(names) * 0.35)))
axes[0].barh(names, counts)
axes[0].invert_yaxis()
axes[0].set_xlabel('document count')
axes[0].set_title('문서 수 (= 풀스캔 시 read 수)')
axes[1].barh(names, kbs, color='tab:orange')
axes[1].invert_yaxis()
axes[1].set_xlabel('total KB')
axes[1].set_title('총 페이로드 크기')
plt.tight_layout()
plt.show()

## STEP 4 — 필드 사용 패턴 분석

- **출현율 100% 가 아닌 필드** = nullable / 일부 문서에만 존재
- **avg/max 크기가 큰 필드** = 매 read마다 같이 다운로드됨 → 분리/요약 고려
- `isHidden`, `projectId` 같은 필드는 인덱스 활용도와 직결됨

In [ ]:
def field_report(collection_name: str, top: int | None = None):
    docs = data.get(collection_name, [])
    if not docs:
        print(f'[{collection_name}] 비어있음')
        return
    n = len(docs)
    counts: Counter = Counter()
    sizes: dict[str, list[int]] = defaultdict(list)
    for d in docs:
        for k, v in d['data'].items():
            counts[k] += 1
            sizes[k].append(len(json.dumps(v, ensure_ascii=False).encode('utf-8')))
    print(f'\n[{collection_name}]  docs={n:,}')
    print(f"  {'field':30s} {'present':>10s} {'%':>6s} {'avg B':>8s} {'p95 B':>8s} {'max B':>8s}")
    print(f"  {'-'*30} {'-'*10} {'-'*6} {'-'*8} {'-'*8} {'-'*8}")
    items = counts.most_common(top)
    for f, c in items:
        s = sorted(sizes[f])
        p95 = s[min(int(len(s)*0.95), len(s)-1)]
        print(f'  {f:30s} {c:>10,} {100*c/n:>5.1f}% {statistics.mean(s):>8,.0f} {p95:>8,.0f} {s[-1]:>8,.0f}')

# 비용 비중이 큰 컬렉션 위주로 살펴보기
for c in ['tasks', 'fa_tasks', 'ict_tasks', 'projects', 'fa_projects', 'ict_projects']:
    if data.get(c):
        field_report(c)

## STEP 5 — projects ↔ tasks 관계 분석

프로젝트 1건 조회 시 `tasks where projectId == X` 가 자주 호출됨.  
→ 프로젝트별 task 수 분포가 read 비용에 직결.

In [ ]:
def relate(projects_col: str, tasks_col: str):
    projs = data.get(projects_col, [])
    tasks = data.get(tasks_col, [])
    if not projs and not tasks:
        return
    proj_map = {d['id']: d['data'] for d in projs}
    by_proj: Counter = Counter()
    orphans = 0
    for t in tasks:
        pid = t['data'].get('projectId')
        by_proj[pid] += 1
        if pid not in proj_map:
            orphans += 1
    hidden_p = sum(1 for d in projs if d['data'].get('isHidden'))
    hidden_t = sum(1 for d in tasks if d['data'].get('isHidden'))

    print(f'\n[{projects_col} ↔ {tasks_col}]')
    print(f'  projects    : {len(projs):,}  (isHidden={hidden_p:,})')
    print(f'  tasks       : {len(tasks):,}  (isHidden={hidden_t:,})')
    if projs:
        print(f'  tasks/proj  : avg={len(tasks)/len(projs):.1f}, max={max(by_proj.values(), default=0):,}')
    if orphans:
        print(f'  ⚠️ orphan tasks (존재하지 않는 projectId 참조): {orphans:,}')
    if by_proj:
        print(f'  top 10 projects by task count:')
        for pid, n in by_proj.most_common(10):
            name = proj_map.get(pid, {}).get('name', '?')
            print(f'    {n:>5,}  [{pid[:8]}…] {name}')

for prefix in ['', 'fa_', 'ict_']:
    relate(f'{prefix}projects' if prefix else 'projects',
           f'{prefix}tasks' if prefix else 'tasks')

## STEP 6 — history 컬렉션 증가 추세

history 는 append-only 라 가장 빠르게 커지는 컬렉션.  
월별 증가량을 보고 보존 정책(예: 90일 이전 삭제) 필요성 검토.

In [ ]:
from datetime import datetime as dt

def parse_dt(v):
    if isinstance(v, dict) and '__datetime__' in v:
        try:
            return dt.fromisoformat(v['__datetime__'].replace('Z', '+00:00'))
        except Exception:
            return None
    return None

for hcol in ['history', 'fa_history', 'ict_history']:
    docs = data.get(hcol, [])
    if not docs:
        continue
    ts = [parse_dt(d['data'].get('createdAt')) for d in docs]
    ts = [t for t in ts if t is not None]
    if not ts:
        print(f'\n[{hcol}] {len(docs):,} docs  (createdAt 없음)')
        continue
    by_month: Counter = Counter(t.strftime('%Y-%m') for t in ts)
    print(f'\n[{hcol}] {len(docs):,} entries  ({min(ts).date()} ~ {max(ts).date()})')
    print('  월별 증가량:')
    for m in sorted(by_month):
        bar = '#' * min(60, by_month[m] // max(1, max(by_month.values()) // 60))
        print(f'    {m}  {by_month[m]:>6,}  {bar}')

## STEP 7 — 일일/월간 read 비용 추정

**전제**: 페이지 1회 진입 시 해당 워크스페이스의 `projects + tasks` 를 풀로드.  
(실제 호출 패턴은 [lib/firestore-service*.ts](../lib) 의 `getProjectsAndTasks*` 함수 기준)

Firestore 가격: **$0.06 / 100K reads** (2026 기준, 리전별 차이 있음)

In [ ]:
PRICE_PER_100K = 0.06       # USD
DAILY_PAGE_LOADS = 100      # ← 실제 트래픽에 맞게 조정
DAYS_PER_MONTH = 30

def reads_per_load(prefix: str, include_hidden: bool = False) -> int:
    pc = f'{prefix}projects' if prefix else 'projects'
    tc = f'{prefix}tasks' if prefix else 'tasks'
    def cnt(col):
        if include_hidden:
            return len(data.get(col, []))
        return sum(1 for d in data.get(col, []) if not d['data'].get('isHidden'))
    return cnt(pc) + cnt(tc)

print(f"전제: 페이지 1회 로드 = projects + tasks 풀스캔, 일 {DAILY_PAGE_LOADS}회 로드\n")
print(f"{'workspace':12s} {'reads/load':>12s} {'reads/day':>12s} {'reads/month':>14s} {'$/month':>10s}")
print('-' * 64)
totals = {'day': 0, 'month': 0, 'cost': 0.0}
for label, prefix in [('strategy', ''), ('fa', 'fa_'), ('ict', 'ict_')]:
    r = reads_per_load(prefix)
    day = r * DAILY_PAGE_LOADS
    month = day * DAYS_PER_MONTH
    cost = month / 100_000 * PRICE_PER_100K
    print(f'{label:12s} {r:>12,} {day:>12,} {month:>14,} {cost:>9.2f}$')
    totals['day'] += day; totals['month'] += month; totals['cost'] += cost
print('-' * 64)
print(f"{'TOTAL':12s} {'':>12s} {totals['day']:>12,} {totals['month']:>14,} {totals['cost']:>9.2f}$")

print('\n👉 최적화 아이디어:')
print('   - isHidden=true 문서가 많다면 Firestore 쿼리에서 server-side filter 가 작동 중인지 확인')
print('   - projects/tasks 분량이 크면 viewport/period 기준 페이징(`where` + `limit`) 도입')
print('   - 자주 변하지 않는 settings, user_profiles 는 localCache + 1회 getDoc 패턴 유지')
print('   - history 는 최근 N건만 onSnapshot, 과거는 별도 lazy 페치')

## STEP 8 — 의심 케이스 자동 탐지

쉽게 잡히는 비용 누수 패턴을 자동 점검.

In [ ]:
ALERTS: list[str] = []

# 1) 매우 큰 단일 문서 (>100KB)
for c, docs in data.items():
    for d in docs:
        b = doc_bytes(d)
        if b > 100 * 1024:
            ALERTS.append(f'BIG DOC  [{c}/{d["id"]}] {b/1024:.1f} KB — 분할 검토')

# 2) 대부분이 isHidden=true 인 컬렉션 (쿼리에 필터 잊고 풀스캔하면 낭비)
for c, docs in data.items():
    if not docs or 'hidden' not in (c + ' '.join(docs[0]['data'].keys())).lower():
        # isHidden 필드를 쓰는 컬렉션만 검사
        if not any('isHidden' in d['data'] for d in docs[:5]):
            continue
    hidden = sum(1 for d in docs if d['data'].get('isHidden'))
    if docs and hidden / len(docs) > 0.5:
        ALERTS.append(f'HIDDEN   [{c}] {hidden:,}/{len(docs):,} ({100*hidden/len(docs):.0f}%) isHidden=true')

# 3) 1000건 초과 컬렉션 — 풀스캔 1회 = 1000+ reads
for c, docs in data.items():
    if len(docs) > 1000:
        ALERTS.append(f'LARGE    [{c}] {len(docs):,} docs — 페이징/필터 필수')

# 4) 큰 배열 필드 (Firestore 1MB 문서 제한, 인덱스 폭증 위험)
for c, docs in data.items():
    for d in docs[:200]:  # 샘플
        for k, v in d['data'].items():
            if isinstance(v, list) and len(v) > 100:
                ALERTS.append(f'BIG ARR  [{c}/{d["id"]}.{k}] len={len(v)}')
                break

if not ALERTS:
    print('✅ 특이사항 없음')
else:
    for a in ALERTS[:50]:
        print(a)
    if len(ALERTS) > 50:
        print(f'… and {len(ALERTS)-50} more')

## STEP 9 — 자유 탐색

여기부터는 자유롭게 셀을 추가해 가설 검증.  
예시: 특정 컬렉션 raw 출력, pandas DataFrame 변환, 시계열 등.

In [ ]:
# 예시: tasks 를 DataFrame으로
import pandas as pd

def to_df(collection_name: str) -> pd.DataFrame:
    docs = data.get(collection_name, [])
    return pd.DataFrame([{'id': d['id'], **d['data']} for d in docs])

df = to_df('tasks')
print(df.shape)
df.head()